# 【夏休み企画】「ラーメンが」の次は「好き」？ Attentionを計算してみよう

対象：日本の高校生を含む、Transformerを初めて学ぶ人  
実行環境：Google Colab無料版（CPUで実行可能）  
題材：中西崇文 [『ChatGPTはどのように動いているのか？』（翔泳社、2026年）](https://amzn.asia/d/09DZPTOy)から発展させた独立教材

このNotebookでは、次の指定文を使います。

> 私がラーメンが[MASK]

「好き」「寝る」「悩む」の三候補をすべてEmbeddingし、入力文のEmbeddingから一層・一ヘッドのSelf-Attentionを計算します。

目標は、Transformer系の言語モデルにも使われる次の計算の一部を、途中の数まで見える大きさで体験することです。

1. トークンをEmbeddingへ変える。
2. QueryとKeyの内積からAttentionを作る。
3. Attentionを重みとしてValueを混ぜる。
4. 混ぜたベクトルと三候補を比べる。
5. 正解例を使って \(W_Q, W_K, W_V\) を学習する。

> **この教材の範囲**  
> このNotebookが作るのは、固定した日本語BERTの単語Embedding表を入力に使う、教材用の三分類模型です。  
> BERTの穴埋め処理や、GPT系LLMの次トークン生成を再現するものではありません。  
> 本物と共通する計算と、省いた計算は、末尾の「本物のLLMとは何が違うか」で切り分けます。


## 著作権と利用条件

このNotebookの説明文と実験コードは、この教材のために新規に作成したものです。  
リポジトリ独自の文章とコードには、同梱の `LICENSE` に示す `All Rights Reserved` と限定的な教育利用許諾が適用されます。  

利用する主な第三者成果物は次のとおりです。

- 東北大学乾研究室の [tohoku-nlp/bert-base-japanese-v3](https://huggingface.co/tohoku-nlp/bert-base-japanese-v3)：Apache License 2.0。TokenizerとトークンEmbedding表を利用します。
- [Transformers](https://github.com/huggingface/transformers)：Apache License 2.0。
- [PyTorch](https://github.com/pytorch/pytorch)：本体はBSD-3-Clause。配布パッケージには別ライセンスの第三者成果物も含まれます。

モデルと各ライブラリには、それぞれのライセンスが別に適用されます。

参考文献とライセンスへのリンクは末尾にもまとめています。


## 0. 実行方法

Google Colabの新しいランタイムで開き、メニューの「ランタイム」→「すべてのセルを実行」を選びます。  
最初の実行では約450 MBの日本語BERTのファイルをダウンロードするため、インターネット接続が必要です。  
GPUは必要ありません。無料版ColabのCPUで動く大きさにしています。

Colabの無償利用枠や使用可能な計算資源は変動します。最新条件は [Google Colab FAQ](https://research.google.com/colaboratory/faq.html) を確認してください。


In [1]:
# Colabの新しいランタイムで必要なライブラリをそろえます。
# バージョンを固定し、公開後も同じ組合せを再現しやすくします。
# このNotebookで使わないColab既定パッケージとの警告は表示しません。
%pip -q install --upgrade --no-warn-conflicts \
    "transformers==5.15.0" \
    "huggingface-hub==1.24.0" \
    "fugashi==1.5.1" \
    "unidic-lite==1.0.8"


In [2]:
import copy
import gc
import logging
import math
import os
import random

# ダウンロードの進捗表示を止め、Notebookへ巨大なwidget情報が残るのを防ぎます。
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
logging.getLogger("torchao.kernel.intmm").setLevel(logging.ERROR)

import huggingface_hub
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from IPython.display import display
from transformers import AutoModelForPreTraining, AutoTokenizer

SEED = 1

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()

# CPUだけで完結させることで、無料版Colabでも同じ条件にします。
DEVICE = torch.device("cpu")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("huggingface-hub:", huggingface_hub.__version__)
print("使用デバイス:", DEVICE)


PyTorch: 2.11.0+cpu
Transformers: 5.15.0
huggingface-hub: 1.24.0
使用デバイス: cpu


## 1. 日本語をトークンとEmbeddingへ変える

コンピュータは「ラーメン」という文字列を、そのまま足したり掛けたりできません。  
そこで、まず文章を**トークン**という単位へ分けます。

次に、各トークンを数の並び（ベクトル）へ変えます。  
この数の並びをEmbedding（埋め込みベクトル）と呼びます。

ベクトルに変えると、二つのトークンを同じ計算式へ入れ、内積やコサイン類似度で「近さ」を数値化できます。  
文字列としては別物である「ラーメン」と「餃子」を、共通する768個の座標で比較できるようになるわけです。

> **本書との対応（紙面ページ）**  
> 34〜38ページ：ベクトルとは何か、言葉をベクトル化する利点。  
> 147〜150ページ：Embeddingの考え方と、各成分を単独では解釈できないという注意。  
> 190ページ：Transformerへ入れる最初の処理としてのInput Embedding。

このNotebookでは、日本語BERTの学習済み**単語Embedding表だけ**を使います。  
BERT Encoder、位置Embedding、文種Embedding、Layer Normalization、Dropoutは使いません。  
BERTで通常加える特殊トークン <code>[CLS]</code> と <code>[SEP]</code> も、この教材では加えません。

数値の尺度をそろえるため、入力トークンと候補のEmbeddingは、長さが1になるよう正規化してから教材模型へ入れます。  
この正規化は教材上の追加処理であり、実際のBERTやGPTで単語Embeddingへ一律に施す標準処理ではありません。


In [3]:
# 本書147〜154ページと190ページに対応する、学習済み単語Embeddingを取得します。
MODEL_NAME = "tohoku-nlp/bert-base-japanese-v3"

print("Tokenizerを読み込んでいます...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Embedding表を取得するため、日本語BERTを一度読み込んでいます...")
pretrained = AutoModelForPreTraining.from_pretrained(MODEL_NAME)

# BERT本体から「単語Embedding表」だけを複製します。
embedding_table = (
    pretrained.bert.embeddings.word_embeddings.weight
    .detach()
    .float()
    .cpu()
    .clone()
)

embedding_dim = embedding_table.shape[1]

# BERTのEncoder本体と学習済み出力層は、この先では使いません。
del pretrained
gc.collect()

print("語彙数:", embedding_table.shape[0])
print("1トークンのEmbedding次元:", embedding_dim)
print("BERTのEncoder本体と学習済み出力層は削除し、単語Embedding表だけを残しました。")


Tokenizerを読み込んでいます...
Embedding表を取得するため、日本語BERTを一度読み込んでいます...
語彙数: 32768
1トークンのEmbedding次元: 768
BERTのEncoder本体と学習済み出力層は削除し、単語Embedding表だけを残しました。


### 1.1 ベクトルにすると「近さ」を計算できる

「ラーメン」「餃子」「桜」を、いま読み込んだ単語Embedding表で768次元ベクトルへ変換します。

文字列のまま比べると、「ラーメン」と「餃子」は同じ文字を含まないため、完全一致か不一致かしか判定できません。  
Embedding後は三語が同じ768次元空間へ置かれるため、ベクトルの向きをコサイン類似度で比較できます。

$$
\operatorname{cosine}(x,y)
=
\frac{x\cdot y}{\lVert x\rVert_2\lVert y\rVert_2}
$$

1に近いほど向きが近く、0に近いほど直交に近いと読みます。  
負の値は、向きが反対側にあることを表します。

> **本書との対応（紙面ページ）**  
> 52〜60ページ：内積、コサイン類似度、色ベクトルによる計算例。  
> 147〜154ページ：Embeddingと、BERTで得た768次元ベクトルの類似度計算。


In [4]:
# 本書52〜60ページ、147〜154ページに対応する実験です。
def make_word_embedding(word):
    # 一語をトークン化し、平均Embeddingを長さ1へ正規化します。
    tokens = tokenizer.tokenize(word)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    vectors = embedding_table[torch.tensor(token_ids, dtype=torch.long)]
    mean_vector = vectors.mean(dim=0)
    return F.normalize(mean_vector, p=2, dim=0), tokens

SIMILARITY_WORDS = ["ラーメン", "餃子", "桜"]
similarity_rows = []
similarity_vectors = []

for word in SIMILARITY_WORDS:
    vector, tokens = make_word_embedding(word)
    similarity_vectors.append(vector)
    similarity_rows.append({
        "語": word,
        "トークン分割": " / ".join(tokens),
        "次元数": vector.numel(),
        "正規化後の長さ": float(vector.norm()),
        "先頭8成分": np.array2string(
            vector[:8].detach().cpu().numpy(),
            precision=5,
            separator=", ",
        ),
    })

similarity_word_x = torch.stack(similarity_vectors)
similarity_matrix = similarity_word_x @ similarity_word_x.T
similarity_df = pd.DataFrame(
    similarity_matrix.detach().cpu().numpy(),
    index=SIMILARITY_WORDS,
    columns=SIMILARITY_WORDS,
)

print("三語を768次元ベクトルへ変換した結果")
display(pd.DataFrame(similarity_rows))

print("\n三語のコサイン類似度")
display(similarity_df.style.format("{:.4f}"))

print(
    "\nラーメンと餃子:",
    f"{similarity_df.loc['ラーメン', '餃子']:.4f}",
)
print(
    "ラーメンと桜  :",
    f"{similarity_df.loc['ラーメン', '桜']:.4f}",
)


三語を768次元ベクトルへ変換した結果


      語   トークン分割  次元数  正規化後の長さ  \
0  ラーメン     ラーメン  768      1.0   
1    餃子  餃 / ##子  768      1.0   
2     桜        桜  768      1.0   

                                               先頭8成分  
0  [-0.04336, -0.02779, -0.04484, -0.06065, -0.02...  
1  [-0.01109,  0.00937, -0.07029,  0.01242, -0.02...  
2  [ 0.04703,  0.03525, -0.04437, -0.00563, -0.02...  


三語のコサイン類似度


,ラーメン,餃子,桜
ラーメン,1.0000,0.3008,0.1082
餃子,0.3008,1.0000,0.0522
桜,0.1082,0.0522,1.0000



ラーメンと餃子: 0.3008
ラーメンと桜  : 0.1082


保存済みの結果では、「ラーメン」と「餃子」の類似度は約0.3008、「ラーメン」と「桜」は約0.1082です。  
このEmbedding表では、「ラーメン」は「桜」よりも「餃子」に近い向きへ配置されています。

この結果は、「ベクトル化すれば、人間が感じる意味の近さを必ず正しく測れる」という保証ではありません。  
この値は、使用した学習済みモデル、トークン分割、複数トークンの平均方法に依存します。

さらに、ここで使っているのは文脈へ入る前の単語Embeddingです。  
同じ語は、どの文章に置いても同じベクトルになります。  
BERT Encoderを通した文脈付きEmbeddingとは区別してください。


### 1.2 指定文と三候補をトークンへ分ける

三語で「ベクトルにすると比較できる」ことを確認したので、指定文と三候補へ進みます。

> **本書との対応（紙面ページ）**  
> 190ページ：Input Embedding。  
> 200〜201ページ：日本語BERTによるトークン化と768次元ベクトル。


In [5]:
DEMO_TEXT = "私がラーメンが[MASK]"
CANDIDATES = ["好き", "寝る", "悩む"]

demo_tokens = tokenizer.tokenize(DEMO_TEXT)

print("入力文:", DEMO_TEXT)
print("トークン:", demo_tokens)
print()

for word in CANDIDATES:
    print(f"{word:>2} -> {tokenizer.tokenize(word)}")


入力文: 私がラーメンが[MASK]
トークン: ['私', 'が', 'ラーメン', 'が', '[MASK]']

好き -> ['好き']
寝る -> ['寝', '##る']
悩む -> ['悩', '##む']


実行結果では、候補によってトークン数が違う場合があります。  
たとえば「寝る」が「寝」と「##る」に分かれたなら、模型は「寝る」を一個の文字列ではなく二個のサブワードとして扱っています。

この教材では、複数トークンに分かれた候補について、そのEmbeddingの平均を取って一個の「候補代表ベクトル」にします。

$$
e_c
=
\operatorname{normalize}
\left(
\frac{1}{m_c}
\sum_{j=1}^{m_c} E_{c,j}
\right)
$$

- $c$：候補。「好き」「寝る」「悩む」のいずれか。
- $m_c$：候補が分かれたトークンの個数。
- $E_{c,j}$：各サブワードのEmbedding。
- $\operatorname{normalize}$：ベクトルの向きは保ち、長さを1にそろえる処理。

これは**教材上の単純化**です。  
二個のトークンを順番に生成する確率を、平均Embeddingで計算できるわけではありません。


In [6]:
MASK_ID = tokenizer.mask_token_id

def normalized_token_embeddings(token_ids):
    ids = torch.tensor(token_ids, dtype=torch.long)
    vectors = embedding_table[ids]
    return F.normalize(vectors, p=2, dim=-1)

def encode_text(text):
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    mask_positions = [i for i, token_id in enumerate(token_ids) if token_id == MASK_ID]
    if len(mask_positions) != 1:
        raise ValueError(f"[MASK]は一つだけ必要です: {text}")
    return {
        "text": text,
        "tokens": tokens,
        "token_ids": token_ids,
        "x": normalized_token_embeddings(token_ids).to(DEVICE),
        "mask_index": mask_positions[0],
    }

candidate_rows = []
candidate_vectors = []

for word in CANDIDATES:
    vector, tokens = make_word_embedding(word)
    candidate_vectors.append(vector)
    candidate_rows.append({
        "候補": word,
        "トークン": " / ".join(tokens),
        "768次元ベクトル": vector.detach().cpu().tolist(),
    })

candidate_x = torch.stack(candidate_vectors).to(DEVICE)
candidate_embedding_768_df = pd.DataFrame(candidate_rows)

assert candidate_embedding_768_df.shape == (len(CANDIDATES), 3)
assert all(
    len(vector) == embedding_dim
    for vector in candidate_embedding_768_df["768次元ベクトル"]
)

summary_df = pd.DataFrame({
    "候補": candidate_embedding_768_df["候補"],
    "トークン": candidate_embedding_768_df["トークン"],
    "次元数": [len(vector) for vector in candidate_embedding_768_df["768次元ベクトル"]],
    "正規化後の長さ": [float(vector.norm()) for vector in candidate_vectors],
})

display(summary_df)

print("\n以下に、各候補の768成分を省略せず表示します。")
print("「寝る」「悩む」は、サブワードのEmbeddingを平均してから正規化しています。")

for row in candidate_rows:
    vector = np.asarray(row["768次元ベクトル"], dtype=np.float32)
    vector_text = np.array2string(
        vector,
        precision=8,
        separator=", ",
        threshold=np.inf,
        max_line_width=120,
    )
    print(f"\n候補: {row['候補']}")
    print(f"トークン: {row['トークン']}")
    print(f"768次元ベクトル（{len(vector)}成分）:")
    print(vector_text)

# 表計算ソフトで一成分ずつ列に分けて確認したい場合に使います。
candidate_embedding_wide_df = pd.DataFrame(
    [
        {
            "候補": row["候補"],
            "トークン": row["トークン"],
            **{
                f"e[{dimension}]": value
                for dimension, value in enumerate(row["768次元ベクトル"], start=1)
            },
        }
        for row in candidate_rows
    ]
)

# CSVへ保存したい場合は、次の行の先頭にある「#」を外してください。
# candidate_embedding_wide_df.to_csv("candidate_embeddings_768.csv", index=False)


   候補     トークン  次元数  正規化後の長さ
0  好き       好き  768      1.0
1  寝る  寝 / ##る  768      1.0
2  悩む  悩 / ##む  768      1.0


以下に、各候補の768成分を省略せず表示します。
「寝る」「悩む」は、サブワードのEmbeddingを平均してから正規化しています。

候補: 好き
トークン: 好き
768次元ベクトル（768成分）:
[ 0.02469859,  0.04778832,  0.01925946, -0.03564764, -0.05840114, -0.07793285, -0.00760291, -0.0018162 , -0.0087839 ,
 -0.0049682 ,  0.00269501,  0.04833198,  0.02878468, -0.06113403,  0.01483806,  0.01320659, -0.04876058, -0.03283902,
 -0.01442603,  0.03875784, -0.11186685, -0.03336274, -0.03126499,  0.04777614, -0.02559669, -0.04371352,  0.06072421,
  0.00099363, -0.06835527, -0.01050096,  0.04045194, -0.01520652, -0.02369317,  0.02219283, -0.04897348, -0.03556847,
  0.03246639, -0.0446394 ,  0.00994571,  0.01384892, -0.05163568, -0.02627584,  0.01562413, -0.02069025,  0.01171242,
  0.01712245, -0.05205621, -0.00889042,  0.04262025,  0.0669392 ,  0.02831734, -0.04832405,  0.01588139, -0.05464088,
 -0.01908554,  0.0973802 , -0.03950341,  0.04082222,  0.01244836,  0.057028  , -0.00111136,  0.00334216, -0.00349432,
  0.02461829, -0.01849862, -0.00280429, -0.01880993,  0.02451097, -0.00

In [7]:
demo = encode_text(DEMO_TEXT)

input_rows = [
    {
        "トークン": token,
        "768次元ベクトル": vector.detach().cpu().tolist(),
    }
    for token, vector in zip(demo["tokens"], demo["x"])
]
input_embedding_768_df = pd.DataFrame(input_rows)

input_summary_df = pd.DataFrame({
    "位置": list(range(len(input_rows))),
    "トークン": input_embedding_768_df["トークン"],
    "次元数": [len(vector) for vector in input_embedding_768_df["768次元ベクトル"]],
    "正規化後の長さ": [float(vector.norm()) for vector in demo["x"]],
})
display(input_summary_df)

print("\n以下に、入力の各トークンの768成分を省略せず表示します。")
for position, row in enumerate(input_rows):
    vector = np.asarray(row["768次元ベクトル"], dtype=np.float32)
    vector_text = np.array2string(
        vector,
        precision=8,
        separator=", ",
        threshold=np.inf,
        max_line_width=120,
    )
    print(f"\n位置: {position}、トークン: {row['トークン']}")
    print(f"768次元ベクトル（{len(vector)}成分）:")
    print(vector_text)

# 表計算ソフトで一成分ずつ列に分けて確認したい場合に使います。
input_embedding_wide_df = pd.DataFrame(
    [
        {
            "位置": position,
            "トークン": row["トークン"],
            **{
                f"e[{dimension}]": value
                for dimension, value in enumerate(row["768次元ベクトル"], start=1)
            },
        }
        for position, row in enumerate(input_rows)
    ]
)

# CSVへ保存したい場合は、次の行の先頭にある「#」を外してください。
# input_embedding_wide_df.to_csv("input_embeddings_768.csv", index=False)


   位置    トークン  次元数  正規化後の長さ
0   0       私  768      1.0
1   1       が  768      1.0
2   2    ラーメン  768      1.0
3   3       が  768      1.0
4   4  [MASK]  768      1.0


以下に、入力の各トークンの768成分を省略せず表示します。

位置: 0、トークン: 私
768次元ベクトル（768成分）:
[-4.19511348e-02,  2.97721010e-02, -4.63242047e-02,  1.43057844e-02, -2.42860783e-02, -2.29983311e-02, -7.26830810e-02,
  5.35152443e-02,  2.07965579e-02,  1.67908445e-02, -9.34059382e-04, -3.40565369e-02,  4.54453602e-02, -4.56828363e-02,
  1.94861321e-03, -1.03876516e-02, -2.14137379e-02, -3.45844589e-02, -8.58142897e-02,  2.47828588e-02, -7.72840902e-02,
  2.79132510e-04,  1.74954273e-02, -1.66331194e-02, -4.99154255e-02,  1.90220717e-02, -1.28863528e-02,  9.36669391e-03,
 -9.01635084e-03,  3.32799666e-02,  3.50228846e-02, -2.26099119e-02, -9.88332927e-03, -7.02732205e-02,  3.55132744e-02,
 -1.39624923e-02, -4.44529392e-02,  2.14254986e-02, -5.59694618e-02,  1.10837165e-02,  1.16406695e-03,  1.28293494e-02,
 -3.09146848e-02,  4.28600609e-02, -8.11436959e-03, -4.47289720e-02,  6.45838678e-02, -1.45716389e-04, -1.14905685e-02,
  4.45946231e-02,  2.13737926e-03,  1.37172658e-02, -3.67363391e-04, -7.98369646e-02,  2.3747459

In [8]:
demo["x"].shape

torch.Size([5, 768])

## 2. 学習例を用意する

模型へ「好き」「寝る」「悩む」の使われ方を教えるため、30件の小さな学習集合を人手で作ります。  
文型をすべて「対象語が[MASK]」にそろえ、助詞や文の長さだけで答える近道を減らします。

| 正解 | 学習に使う対象語 | 検証に使う別の対象語 |
|---|---|---|
| 好き | 寿司、カレー、餃子、音楽、映画、旅行、読書、珈琲、ケーキ、写真 | うどん、焼肉、漫画 |
| 寝る | 赤ちゃん、乳児、子猫、子犬、弟、妹、父、母、祖父、祖母 | 幼児、子ども、飼い犬 |
| 悩む | 受験生、就活生、経営者、担当者、患者、保護者、学生、部長、先生、研究者 | 浪人生、責任者、管理職 |

「ラーメン」は、どちらにも入れません。  
ただし、これらは研究用に無作為抽出した大規模データではありません。  
人間が教材用に選んだ例なので、検証正解率を一般的な日本語能力の指標にはできません。


In [9]:
TRAIN_SUBJECTS = {
    "好き": ["寿司", "カレー", "餃子", "音楽", "映画", "旅行", "読書", "珈琲", "ケーキ", "写真"],
    "寝る": ["赤ちゃん", "乳児", "子猫", "子犬", "弟", "妹", "父", "母", "祖父", "祖母"],
    "悩む": ["受験生", "就活生", "経営者", "担当者", "患者", "保護者", "学生", "部長", "先生", "研究者"],
}

VALID_SUBJECTS = {
    "好き": ["うどん", "焼肉", "漫画"],
    "寝る": ["幼児", "子ども", "飼い犬"],
    "悩む": ["浪人生", "責任者", "管理職"],
}

candidate_to_id = {word: i for i, word in enumerate(CANDIDATES)}

def build_dataset(subject_groups):
    items = []
    for answer, subjects in subject_groups.items():
        for subject in subjects:
            item = encode_text(f"{subject}が[MASK]")
            item["label"] = candidate_to_id[answer]
            item["answer"] = answer
            item["subject"] = subject
            items.append(item)
    return items

train_data = build_dataset(TRAIN_SUBJECTS)
valid_data = build_dataset(VALID_SUBJECTS)

print("学習例:", len(train_data), "件")
print("検証例:", len(valid_data), "件")
print("例:", train_data[0]["text"], "→", train_data[0]["answer"])


学習例: 30 件
検証例: 9 件
例: 寿司が[MASK] → 好き


## 3. 一層・一ヘッドのSelf-Attentionを作る

> **本書との対応（紙面ページ）**  
> 194〜199ページ：Attentionの式、Q・K・V、softmax、\(\sqrt{d_k}\)による尺度調整。  
> 200〜203ページ：日本語BERTを使ったSelf-Attentionの計算例。

ここからが中心です。  
768次元のEmbeddingを、目で追える4次元のQuery、Key、Valueへ変換します。

入力を縦に並べた行列を $X$ とし、学習する三個の行列を $W_Q,W_K,W_V$ とします。

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V
$$

この教材では、各行列の形は $768\times4$ です。

- Query \(Q\)：どのような手掛かりを探すか。
- Key \(K\)：各トークンがどのような手掛かりを持つか。
- Value \(V\)：見つけた場所から実際に取り込む中身。

これは役割をつかむための説明です。  
一個一個の数に「食べ物らしさ」などの固定した意味が書かれているわけではありません。

### 3.1 QueryとKeyの内積

<code>[MASK]</code>位置のQueryを $q_{\text{mask}}$、入力の $i$番目のKeyを$k_i$ とします。

$$
q_{\text{mask}}\cdot k_i
=
q_1k_{i,1}+q_2k_{i,2}+q_3k_{i,3}+q_4k_{i,4}
$$

これは、二つの4次元ベクトルの成分を一組ずつ掛け、全部足す計算です。  
値が相対的に大きいトークンほど、現在の $W_Q,W_K$ が作った空間で「このQueryと合うKey」と判定されます。

ここでいう「合う」は、日常語の意味が似ているという保証ではありません。  
**学習されたAttention上の相性**です。

### 3.2 $\sqrt{d}$ で割る

Scaled Dot-Product Attentionでは、内積を $\sqrt{d}$ で割ります。

$$
s_i=\frac{q_{\text{mask}}\cdot k_i}{\sqrt{d}}
$$

今回は $d=4$ なので、$\sqrt{4}=2$ です。  
次元数が多いほど内積の絶対値が大きくなりやすいため、極端になりすぎないよう尺度を調整します。

### 3.3 softmaxで割合へ変える

$$
a_i
=
\frac{\exp(s_i)}
{\sum_t \exp(s_t)}
$$

$\exp$ は指数関数です。  
softmaxを通すと、すべての $a_i$ は0より大きくなり、合計は1、つまり100%になります。

### 3.4 Valueを重み付き平均する

$$
h_{\text{mask}}
=
\sum_i a_i v_i
$$

Attention $a_i$ が大きいトークンのValue$v_i$は多く、小さいトークンのValueは少なく混ざります。  
これが、このNotebookで観察するSelf-Attentionの中心計算です。


In [10]:
# 本書194〜203ページの計算を、学習可能なW_Q、W_K、W_Vで実装します。
class TinySelfAttentionClassifier(nn.Module):
    """
    教材用の一層・一ヘッドSelf-Attention模型。

    学習するもの:
      W_Q, W_K, W_V（すべて 768 -> 4、biasなし）

    固定するもの:
      日本語BERTのEmbedding表から作った入力と候補Embedding
    """
    def __init__(self, candidate_embeddings, input_dim=768, attention_dim=4):
        super().__init__()
        self.attention_dim = attention_dim
        self.w_q = nn.Linear(input_dim, attention_dim, bias=False)
        self.w_k = nn.Linear(input_dim, attention_dim, bias=False)
        self.w_v = nn.Linear(input_dim, attention_dim, bias=False)
        self.register_buffer("candidate_x", candidate_embeddings.clone())

        # 学習前のAttentionがほぼ均等になる小さな乱数で始めます。
        for layer in (self.w_q, self.w_k, self.w_v):
            nn.init.normal_(layer.weight, mean=0.0, std=0.02)

    def forward(self, x, mask_index):
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        raw_scores = q @ k.T
        scaled_scores = raw_scores / math.sqrt(self.attention_dim)
        attention = torch.softmax(scaled_scores, dim=-1)
        h = attention @ v

        h_mask = h[mask_index]

        # 候補も同じEmbedding表から作り、入力と同じW_Vへ通します。
        candidate_v = self.w_v(self.candidate_x)

        # h_maskと各候補の角度の近さ（コサイン類似度）を得点にします。
        logits = F.cosine_similarity(
            h_mask.unsqueeze(0),
            candidate_v,
            dim=-1,
        )

        return {
            "logits": logits,
            "q": q,
            "k": k,
            "v": v,
            "raw_scores": raw_scores,
            "scaled_scores": scaled_scores,
            "attention": attention,
            "h": h,
            "h_mask": h_mask,
            "candidate_v": candidate_v,
        }

set_seed(SEED)
model = TinySelfAttentionClassifier(
    candidate_embeddings=candidate_x,
    input_dim=embedding_dim,
    attention_dim=4,
).to(DEVICE)

before_training = copy.deepcopy(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("学習可能パラメータ数:", f"{trainable:,}")
print(
    f"内訳: {embedding_dim} × {model.attention_dim} × 3 =",
    embedding_dim * model.attention_dim * 3,
)


学習可能パラメータ数: 9,216
内訳: 768 × 4 × 3 = 9216


30件の学習例に対し、学習する数は9,216個あります。  
したがって、学習例へ100%正解しても不思議ではありません。  
それだけで、広い日本語に通用する規則を発見したとは言えません。

また、この模型には位置情報がありません。  
同じトークン「が」が二回あれば、同じEmbeddingから同じKeyとValueが作られます。  
二個の「が」を位置で区別できないことは、後で実行結果から確かめます。


## 4. 文脈ベクトルと三候補を比べる

> **本書との対応（紙面ページ）**  
> 52〜60ページ：コサイン類似度。  
> 192〜193ページ：出力候補とsoftmax。  
> このNotebookは語彙全体へのLinear層を使わず、三候補だけを比較します。

Self-Attentionが作った $h_{\text{mask}}$ は、Valueと同じ4次元空間にあります。  
そこで、候補Embedding $e_c$ にも**同じ** $W_V$ を掛けます。

$$
v_c=e_cW_V
$$

文脈と候補のコサイン類似度を、候補の得点 $z_c$ にします。

$$
z_c
=
\cos(h_{\text{mask}},v_c)
=
\frac{h_{\text{mask}}\cdot v_c}
{\lVert h_{\text{mask}}\rVert_2\lVert v_c\rVert_2}
$$

向きが同じなら1、直角なら0、反対向きなら-1に近づきます。

表示を見やすくするため、三得点だけにsoftmaxを掛けます。

$$
r_c
=
\frac{\exp(z_c)}
{\sum_{j\in\{\text{好き,寝る,悩む}\}}\exp(z_j)}
$$

ここには重要な限定があります。

- $r_c$ は三候補の中だけで比べた表示用の比率です。
- 語彙全体に対する生成確率ではありません。
- BERTやChatGPTが返した確率でもありません。
- コサイン得点は $[-1,1]$ に限られるため、温度係数なしでは一候補の比率は最大でも約78.7%です。

出力専用の $W_{\mathrm{OUT}}$ は置いていません。  
ただし、恣意性が消えたわけではありません。  
$W_V$ は「入力から取り込むValueを作る役割」と「候補を比較空間へ移す役割」の二つを兼ねています。


In [11]:
def result_tables(current_model, text):
    item = encode_text(text)
    current_model.eval()

    with torch.no_grad():
        out = current_model(item["x"], item["mask_index"])
        mask_index = item["mask_index"]
        q_mask = out["q"][mask_index]

        scale_label = f"÷ √{current_model.attention_dim} 後"
        attention_rows = []
        for i, token in enumerate(item["tokens"]):
            attention_rows.append({
                "位置": i,
                "トークン": token,
                "内積 q・k": float(out["raw_scores"][mask_index, i]),
                scale_label: float(out["scaled_scores"][mask_index, i]),
                "Attention": float(out["attention"][mask_index, i]),
            })

        ratios = torch.softmax(out["logits"], dim=0)
        candidate_rows = []
        for i, word in enumerate(CANDIDATES):
            candidate_rows.append({
                "候補": word,
                "候補のトークン分割": " / ".join(tokenizer.tokenize(word)),
                "コサイン得点": float(out["logits"][i]),
                "三候補内の比率": float(ratios[i]),
            })

    return item, out, pd.DataFrame(attention_rows), pd.DataFrame(candidate_rows)

def show_result(current_model, text):
    item, out, attention_df, candidate_df = result_tables(current_model, text)

    print("入力:", text)
    print("トークン:", item["tokens"])
    print("[MASK]位置のQuery:", np.round(out["q"][item["mask_index"]].detach().numpy(), 4))
    print("[MASK]位置の文脈ベクトル:", np.round(out["h_mask"].detach().numpy(), 4))
    print()

    scale_label = f"÷ √{current_model.attention_dim} 後"
    print("【Q・KからAttentionを作る】")
    display(
        attention_df.style
        .format({
            "内積 q・k": "{:.5f}",
            scale_label: "{:.5f}",
            "Attention": "{:.1%}",
        })
        .bar(subset=["Attention"], vmin=0, vmax=1, color="#80bfff")
    )
    print("Attention合計:", f"{attention_df['Attention'].sum():.6f}")
    print()

    print("【文脈ベクトルと三候補を比べる】")
    display(
        candidate_df.sort_values("三候補内の比率", ascending=False).style
        .format({
            "コサイン得点": "{:.5f}",
            "三候補内の比率": "{:.1%}",
        })
        .bar(subset=["三候補内の比率"], vmin=0, vmax=1, color="#ffb366")
    )

    winner = candidate_df.loc[candidate_df["三候補内の比率"].idxmax(), "候補"]
    print("一位:", winner)
    return item, out, attention_df, candidate_df


## 5. 学習前の計算を観察する

まだ正解例を一度も使っていません。  
\(W_Q,W_K,W_V\) は小さな乱数なので、Attentionはほぼ均等になるはずです。  
学習前の三候補の一位は乱数による偶然であり、意味のある予測ではありません。

ここでは、次の二種類の「近さ」を分けて見てください。

1. **Q–Kの内積**：入力中のどのValueをどの割合で取り込むかを決める相性。
2. **文脈–候補のコサイン**：「好き」「寝る」「悩む」の順位を決める相性。

Q–Kの内積が、直接「好き」の得点になるわけではありません。


In [12]:
before_item, before_out, before_attention, before_candidates = show_result(
    before_training,
    DEMO_TEXT,
)


入力: 私がラーメンが[MASK]
トークン: ['私', 'が', 'ラーメン', 'が', '[MASK]']
[MASK]位置のQuery: [-0.0298 -0.0069  0.0063 -0.0253]
[MASK]位置の文脈ベクトル: [-0.0139 -0.012  -0.0013 -0.0008]

【Q・KからAttentionを作る】


,位置,トークン,内積 q・k,÷ √4 後,Attention
0,0,私,0.00043,0.00022,20.0%
1,1,が,0.00028,0.00014,20.0%
2,2,ラーメン,0.00006,0.00003,20.0%
3,3,が,0.00028,0.00014,20.0%
4,4,[MASK],-0.00170,-0.00085,20.0%


Attention合計: 1.000000

【文脈ベクトルと三候補を比べる】


,候補,候補のトークン分割,コサイン得点,三候補内の比率
2,悩む,悩 / ##む,0.19969,41.9%
0,好き,好き,-0.09209,31.3%
1,寝る,寝 / ##る,-0.24415,26.9%


一位: 悩む


### 内積を本当に4回の掛け算で確かめる

<code>[MASK]</code>のQueryと「ラーメン」のKeyを選び、内積を成分ごとに展開します。  
Tokenizerの結果によって「ラーメン」が複数トークンに分かれた場合は、最初に「ラーメン」を含むトークンを使います。


In [13]:
def explain_dot_product(current_model, text, token_keyword="ラーメン"):
    item = encode_text(text)
    current_model.eval()
    with torch.no_grad():
        out = current_model(item["x"], item["mask_index"])

    target_positions = [
        i for i, token in enumerate(item["tokens"])
        if token_keyword in token.replace("##", "")
    ]
    if not target_positions:
        target_positions = [0]

    i = target_positions[0]
    q = out["q"][item["mask_index"]]
    k = out["k"][i]
    products = q * k

    print("選んだトークン:", item["tokens"][i])
    print("q_mask =", np.round(q.numpy(), 6))
    print("k_i    =", np.round(k.numpy(), 6))
    print()
    for j in range(len(products)):
        print(
            f"成分{j+1}: {q[j].item(): .6f} × {k[j].item(): .6f}"
            f" = {products[j].item(): .6f}"
        )
    print("-" * 48)
    scale = math.sqrt(current_model.attention_dim)
    print(f"内積（{len(products)}項の和） =", f"{products.sum().item(): .6f}")
    print(
        f"√{current_model.attention_dim}で割った値   =",
        f"{(products.sum()/scale).item(): .6f}",
    )
    print(
        "表の値          =",
        f"{out['scaled_scores'][item['mask_index'], i].item(): .6f}",
    )

explain_dot_product(before_training, DEMO_TEXT)


選んだトークン: ラーメン
q_mask = [-0.029759 -0.006914  0.006345 -0.025338]
k_i    = [ 0.014133 -0.020719 -0.007738 -0.015146]

成分1: -0.029759 ×  0.014133 = -0.000421
成分2: -0.006914 × -0.020719 =  0.000143
成分3:  0.006345 × -0.007738 = -0.000049
成分4: -0.025338 × -0.015146 =  0.000384
------------------------------------------------
内積（4項の和） =  0.000057
√4で割った値   =  0.000029
表の値          =  0.000029


## 6. 正解例から三行列を学習する

> **本書との対応（紙面ページ）**  
> 120〜132ページ：正解との誤差を手掛かりに、行列を少しずつ直すバックプロパゲーション。  
> 本書のじゃんけん模型とは課題と損失関数が異なりますが、「誤差から行列を更新する」という考え方は共通します。

### 6.1 「ラーメンなら好き」と直接教えてはいない

学習データには「ラーメン」を入れていません。  
模型へ与えるのは、次のような30件の正解付き例です。

- 「私が寿司が[MASK]」の正解は「好き」。
- 「私が赤ちゃんが[MASK]」の正解は「寝る」。
- 「私が受験生が[MASK]」の正解は「悩む」。

したがって、模型が「ラーメン」を丸暗記することはできません。  
学習後に「ラーメン」から「好き」が一位になるなら、学習済みEmbeddingに含まれる関係と、30件から学んだ三行列を組み合わせた結果です。

### 6.2 一回の更新で起きること

一つの学習例について、模型が三候補へ与えた得点を \(z_0,z_1,z_2\) とします。  
正解候補を \(y\) とすると、損失は交差エントロピーで測ります。

$$
L
=
-\log
\left(
\frac{\exp(z_y)}
{\exp(z_0)+\exp(z_1)+\exp(z_2)}
\right)
$$

「寿司」の例で「好き」の比率が低ければ、損失 \(L\) は大きくなります。  
PyTorchの自動微分は、「どの数をどちらへ動かせば損失が下がるか」を \(W_Q,W_K,W_V\) の各成分について計算します。  
AdamWは、その方向へ三行列を少しだけ更新します。

このNotebookでは、30件をまとめて一回計算する処理を200回繰り返します。  
人間が「ラーメンへ注目する値」や「好きの得点を上げる値」を行列へ直接書き込んでいるわけではありません。

三行列の役割は、次のように分けられます。

- \(W_Q,W_K\)：<code>[MASK]</code>が入力中のどのトークンからValueを多く受け取るかを変える。
- \(W_V\)：入力トークンのValueと、比較対象となる候補ベクトルの向きを同時に変える。

三行列は一緒に最適化されるため、「好きになった原因は \(W_Q\) だけ」のようには分解できません。  
とくに \(W_V\) は入力と候補の双方に使うという、この教材固有の設計です。

学習では、次を繰り返します。

1. 30件の得点と損失を計算する。
2. 損失を小さくする方向を自動微分で求める。
3. \(W_Q,W_K,W_V\) の9,216個の数を少し直す。
4. 学習に使っていない9件の検証損失も確認する。

更新法にはAdamWを使います。  
このNotebookの中心は更新法そのものではなく、更新前後でAttentionと候補順位がどう変わるかです。


In [14]:
# 本書120〜132ページの『誤差を使って行列を更新する』流れに対応します。
def stack_logits(current_model, dataset):
    return torch.stack([
        current_model(item["x"], item["mask_index"])["logits"]
        for item in dataset
    ])

def dataset_metrics(current_model, dataset):
    current_model.eval()
    labels = torch.tensor([item["label"] for item in dataset], device=DEVICE)
    with torch.no_grad():
        logits = stack_logits(current_model, dataset)
        loss = F.cross_entropy(logits, labels)
        accuracy = (logits.argmax(dim=1) == labels).float().mean()
    return float(loss), float(accuracy)

def fit_model(current_model, train_dataset, valid_dataset, steps=200, learning_rate=5e-3):
    labels = torch.tensor([item["label"] for item in train_dataset], device=DEVICE)
    optimizer = torch.optim.AdamW(
        current_model.parameters(),
        lr=learning_rate,
        weight_decay=1e-3,
    )

    best_valid_loss = float("inf")
    best_state = None
    best_step = 0
    history = []

    for step in range(1, steps + 1):
        current_model.train()
        optimizer.zero_grad()

        logits = stack_logits(current_model, train_dataset)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()

        if step == 1 or step % 5 == 0:
            train_loss, train_acc = dataset_metrics(current_model, train_dataset)
            valid_loss, valid_acc = dataset_metrics(current_model, valid_dataset)

            history.append({
                "step": step,
                "学習損失": train_loss,
                "学習正解率": train_acc,
                "検証損失": valid_loss,
                "検証正解率": valid_acc,
            })

            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                best_state = copy.deepcopy(current_model.state_dict())
                best_step = step

        if step == 1 or step % 100 == 0 or step == steps:
            print(
                f"step {step:>3}: "
                f"学習損失={train_loss:.4f}, 学習正解率={train_acc:.1%}, "
                f"検証損失={valid_loss:.4f}, 検証正解率={valid_acc:.1%}"
            )

    current_model.load_state_dict(best_state)
    print(f"\n検証損失が最小だったstep {best_step}の状態を採用しました。")
    return pd.DataFrame(history)

history = fit_model(model, train_data, valid_data)


step   1: 学習損失=1.1131, 学習正解率=53.3%, 検証損失=1.2056, 検証正解率=44.4%
step 100: 学習損失=0.3690, 学習正解率=100.0%, 検証損失=0.4394, 検証正解率=100.0%
step 200: 学習損失=0.3690, 学習正解率=100.0%, 検証損失=0.4399, 検証正解率=100.0%

検証損失が最小だったstep 65の状態を採用しました。


In [15]:
print("採用した模型の成績")
train_loss, train_acc = dataset_metrics(model, train_data)
valid_loss, valid_acc = dataset_metrics(model, valid_data)

metrics_df = pd.DataFrame([
    {"データ": "学習30件", "損失": train_loss, "正解率": train_acc},
    {"データ": "検証9件", "損失": valid_loss, "正解率": valid_acc},
])

display(
    metrics_df.style
    .format({"損失": "{:.4f}", "正解率": "{:.1%}"})
    .bar(subset=["正解率"], vmin=0, vmax=1, color="#8fd18f")
)

print("注意：小規模で人手選択したデータへの成績です。一般的な日本語性能ではありません。")


採用した模型の成績


,データ,損失,正解率
0,学習30件,0.3694,100.0%
1,検証9件,0.4322,100.0%


注意：小規模で人手選択したデータへの成績です。一般的な日本語性能ではありません。


## 7. 学習後のAttentionと候補順位を見る

> **本書との対応（紙面ページ）**  
> 147〜154ページ：Embedding上の近さ。  
> 194〜203ページ：Q・K・VからAttentionを計算する流れ。

学習後も、計算式はまったく同じです。  
変わったのは \(W_Q,W_K,W_V\) の中の数です。

学習後の表で確かめることは三点です。

1. Q–K内積とAttentionは学習前から変わったか。
2. Attentionの合計は100%になっているか。
3. 三候補では何が一位になったか。


In [16]:
after_item, after_out, after_attention, after_candidates = show_result(
    model,
    DEMO_TEXT,
)


入力: 私がラーメンが[MASK]
トークン: ['私', 'が', 'ラーメン', 'が', '[MASK]']
[MASK]位置のQuery: [-0.5747 -0.5547  0.5706 -0.5809]
[MASK]位置の文脈ベクトル: [-0.0111 -0.0997 -0.0753 -0.1308]

【Q・KからAttentionを作る】


,位置,トークン,内積 q・k,÷ √4 後,Attention
0,0,私,0.03357,0.01679,24.9%
1,1,が,-2.03174,-1.01587,8.9%
2,2,ラーメン,0.93100,0.46550,39.1%
3,3,が,-2.03174,-1.01587,8.9%
4,4,[MASK],-0.59079,-0.29540,18.2%


Attention合計: 1.000000

【文脈ベクトルと三候補を比べる】


,候補,候補のトークン分割,コサイン得点,三候補内の比率
0,好き,好き,0.97543,68.7%
2,悩む,悩 / ##む,-0.48305,16.0%
1,寝る,寝 / ##る,-0.52131,15.4%


一位: 好き


### 7.1 なぜ、未学習の「ラーメン」から「好き」が出たのか

「ラーメン」は30件の学習語に含まれていません。  
そこで、まず学習済み単語Embeddingの段階で、「ラーメン」が30語のどれに近いかをすべて計算します。

この表は、都合のよい語だけを後から選ぶのではなく、30語全部を計算して近い順に並べます。


In [17]:
# 本書147〜154ページのEmbeddingとコサイン類似度に対応します。
ramen_vector, _ = make_word_embedding("ラーメン")
ramen_neighbor_rows = []

for correct_candidate, words in TRAIN_SUBJECTS.items():
    for word in words:
        word_vector, tokens = make_word_embedding(word)
        ramen_neighbor_rows.append({
            "学習語": word,
            "トークン分割": " / ".join(tokens),
            "学習時の正解": correct_candidate,
            "ラーメンとのコサイン類似度": float(torch.dot(ramen_vector, word_vector)),
        })

ramen_neighbor_df = (
    pd.DataFrame(ramen_neighbor_rows)
    .sort_values("ラーメンとのコサイン類似度", ascending=False)
    .reset_index(drop=True)
)
ramen_neighbor_df.insert(0, "順位", np.arange(1, len(ramen_neighbor_df) + 1))

print("ラーメンに近い学習語の上位10件（30語すべてを計算）")
display(
    ramen_neighbor_df.head(10).style.format({
        "ラーメンとのコサイン類似度": "{:.4f}",
    })
)


ラーメンに近い学習語の上位10件（30語すべてを計算）


,順位,学習語,トークン分割,学習時の正解,ラーメンとのコサイン類似度
0,1,カレー,カレー,好き,0.3730
1,2,寿司,寿司,好き,0.3242
2,3,餃子,餃 / ##子,好き,0.3008
3,4,珈琲,珈 / ##琲,好き,0.2725
4,5,ケーキ,ケーキ,好き,0.2608
5,6,祖母,祖母,寝る,0.1627
6,7,患者,患者,悩む,0.1625
7,8,赤ちゃん,赤ちゃん,寝る,0.1509
8,9,部長,部長,悩む,0.1189
9,10,先生,先生,悩む,0.0969


保存済み結果では、上位5語が「カレー」「寿司」「餃子」「珈琲」「ケーキ」となり、学習時の正解はいずれも「好き」です。  
事前学習済みEmbeddingが、「ラーメン」を学習済みの食べ物に近い場所へ置いていたことが、未学習語へ結果を移す足場になっています。

ただし、最終予測は単純な最近傍検索ではありません。  
模型は入力文全体を \(W_Q,W_K,W_V\) へ通し、<code>[MASK]</code>位置の文脈ベクトルを作ってから三候補と比べています。

保存済み結果では、「ラーメン」へのAttentionが学習前の20.0%から学習後の39.1%へ上がり、三候補では「好き」が68.7%で一位になります。  
この変化は、学習によって三行列が変わった事実を示します。

しかし、後の均等Attention対照も同じ小規模課題を解けています。  
したがって、「ラーメンへのAttentionが39.1%になったから、好きと答えられた」という因果までは証明できません。  
この課題では、事前学習済みEmbeddingと \(W_V\) の学習だけでも、かなりの部分を解けるからです。


In [18]:
comparison = before_attention[["位置", "トークン", "Attention"]].copy()
comparison = comparison.rename(columns={"Attention": "学習前"})
comparison["学習後"] = after_attention["Attention"]
comparison["変化"] = comparison["学習後"] - comparison["学習前"]

display(
    comparison.style
    .format({
        "学習前": "{:.1%}",
        "学習後": "{:.1%}",
        "変化": "{:+.1%}",
    })
    .bar(subset=["学習後"], vmin=0, vmax=1, color="#80bfff")
)


,位置,トークン,学習前,学習後,変化
0,0,私,20.0%,24.9%,+4.9%
1,1,が,20.0%,8.9%,-11.1%
2,2,ラーメン,20.0%,39.1%,+19.1%
3,3,が,20.0%,8.9%,-11.1%
4,4,[MASK],20.0%,18.2%,-1.7%


In [19]:
print("学習後の内積も、4回の掛け算へ戻って確認します。")
explain_dot_product(model, DEMO_TEXT)


学習後の内積も、4回の掛け算へ戻って確認します。
選んだトークン: ラーメン
q_mask = [-0.574682 -0.554746  0.570631 -0.580864]
k_i    = [-0.404219 -0.413232  0.384447 -0.430548]

成分1: -0.574682 × -0.404219 =  0.232297
成分2: -0.554746 × -0.413232 =  0.229239
成分3:  0.570631 ×  0.384447 =  0.219377
成分4: -0.580864 × -0.430548 =  0.250090
------------------------------------------------
内積（4項の和） =  0.931003
√4で割った値   =  0.465502
表の値          =  0.465502


### ここで何を「体感」できたか

Q–K内積が大きい場所ほど、softmax後のAttentionも大きくなります。  
この順序は、式から必ず成り立ちます。

$$
q_{\text{mask}}\cdot k_i
\rightarrow
\frac{q_{\text{mask}}\cdot k_i}{\sqrt{4}}
\rightarrow
a_i
\rightarrow
h_{\text{mask}}=\sum_i a_iv_i
$$

ただし、Attentionが大きいトークンを見つけただけでは、最終候補は決まりません。  
その場所のValueが何を運ぶか、他の場所のValueとどう混ざるか、文脈ベクトルが候補ベクトルとどの向きになるかも関係します。

したがって、

> 学習されたQとKの内積が、入力中のどのValueを多く取り込むかという「相性」を作った。


## 8. 対象語だけを変える比較実験

同じ学習済み模型へ、三つの入力を与えます。

- 私がラーメンが[MASK]
- 私が幼児が[MASK]
- 私が浪人生が[MASK]

模型の再学習はしません。  
変えるのは対象語だけです。


In [20]:
def compact_prediction(current_model, text):
    item, out, attention_df, candidate_df = result_tables(current_model, text)
    target_tokens = [
        (token, float(attention_df.loc[i, "Attention"]))
        for i, token in enumerate(item["tokens"])
    ]
    winner_row = candidate_df.loc[candidate_df["三候補内の比率"].idxmax()]
    return {
        "入力": text,
        "トークン": " / ".join(item["tokens"]),
        "Attention最大": max(target_tokens, key=lambda pair: pair[1])[0],
        "最大Attention": max(pair[1] for pair in target_tokens),
        "一位": winner_row["候補"],
        "一位の比率": winner_row["三候補内の比率"],
    }

comparison_texts = [
    "私がラーメンが[MASK]",
    "私が幼児が[MASK]",
    "私が浪人生が[MASK]",
]

experiment_df = pd.DataFrame([
    compact_prediction(model, text)
    for text in comparison_texts
])

display(
    experiment_df.style.format({
        "最大Attention": "{:.1%}",
        "一位の比率": "{:.1%}",
    })
)


,入力,トークン,Attention最大,最大Attention,一位,一位の比率
0,私がラーメンが[MASK],私 / が / ラーメン / が / [MASK],ラーメン,39.1%,好き,68.7%
1,私が幼児が[MASK],私 / が / 幼児 / が / [MASK],幼児,41.1%,寝る,68.0%
2,私が浪人生が[MASK],私 / が / 浪人 / 生 / が / [MASK],浪人,29.3%,悩む,66.0%


結果が期待どおりでも、「三個の文だけで一般化を証明した」とは書けません。  
夏休みの自由研究なら、対象語を自分で10個以上決め、最初に予想を書いてから結果と比較してください。

観察記録は、次の三文に分けると明確です。

1. **事実**：「○○では、候補△△が一位だった。」
2. **解釈**：「学習例の□□とEmbedding上の特徴が近かった可能性がある。」
3. **限界**：「三候補だけの教材模型なので、一般的な日本語生成能力は評価していない。」


## 9. 位置情報がないことを確かめる

指定文には「が」が二回あります。  
この模型には位置Embeddingがないため、二個の「が」は同じEmbedding、同じKey、同じValueになります。

<code>[MASK]</code>から見た二個の「が」のAttentionも一致するはずです。


In [21]:
ga_rows = after_attention[after_attention["トークン"] == "が"][
    ["位置", "トークン", "内積 q・k", "Attention"]
]

display(
    ga_rows.style.format({
        "内積 q・k": "{:.8f}",
        "Attention": "{:.8f}",
    })
)

if len(ga_rows) == 2:
    same = np.allclose(
        ga_rows.iloc[0][["内積 q・k", "Attention"]].astype(float),
        ga_rows.iloc[1][["内積 q・k", "Attention"]].astype(float),
    )
    print("二個の「が」の値は一致したか:", same)
else:
    print("Tokenizerの分割結果で「が」が二個にならなかったため、個数を確認してください。")


,位置,トークン,内積 q・k,Attention
1,1,が,-2.03174067,0.08878709
3,3,が,-2.03174067,0.08878709


二個の「が」の値は一致したか: True


この一致はバグではなく、位置情報を入れていない模型の限界です。  
実際のTransformer系モデルは位置情報を加え、同じトークンでも置かれた位置を区別できるようにします。

次の発展編で位置Embeddingを追加するときは、この「二個の『が』を区別できない」という不便から始めると、部品を足す理由がわかりやすくなります。


## 10. 自然な文との比較

指定文「私がラーメンが[MASK]」は、主題と主語の置き方が自然ではありません。  
そこで「私はラーメンが[MASK]」とも比べます。

この模型には位置情報も文法を処理する多数の層もありません。  
結果が同じでも違っても、助詞「が」と「は」の一般的な働きを理解した証拠にはなりません。


In [22]:
grammar_comparison = pd.DataFrame([
    compact_prediction(model, "私がラーメンが[MASK]"),
    compact_prediction(model, "私はラーメンが[MASK]"),
    compact_prediction(model, "ラーメンが[MASK]"),
])

display(
    grammar_comparison.style.format({
        "最大Attention": "{:.1%}",
        "一位の比率": "{:.1%}",
    })
)


,入力,トークン,Attention最大,最大Attention,一位,一位の比率
0,私がラーメンが[MASK],私 / が / ラーメン / が / [MASK],ラーメン,39.1%,好き,68.7%
1,私はラーメンが[MASK],私 / は / ラーメン / が / [MASK],ラーメン,37.5%,好き,68.6%
2,ラーメンが[MASK],ラーメン / が / [MASK],ラーメン,59.0%,好き,68.8%


## 11. 厳しい対照実験：Attentionを均等にしても解けるか

ここまでのグラフを見ると、「Attentionが正解を生んだ」と言いたくなります。  
その主張を確かめるため、QとKを使わず、すべての入力Valueを同じ割合で混ぜる模型も学習します。

入力が5トークンなら、各Attentionは20%です。

$$
a_i=\frac{1}{5}
$$

この対照模型でも $W_V$ は学習します。  
もし同じ三分類を解けたなら、この小さなデータでは「学習されたAttentionが必要だった」とは証明できません。

この実験はAttentionが無意味だと示すものでもありません。  
**見えたAttentionと、出力に不可欠な原因とは同じとは限らない**と確認するための対照です。


In [23]:
class UniformAttentionClassifier(nn.Module):
    """QとKを持たず、全トークンのValueを均等に混ぜる対照模型。"""
    def __init__(self, candidate_embeddings, input_dim=768, attention_dim=4):
        super().__init__()
        self.w_v = nn.Linear(input_dim, attention_dim, bias=False)
        self.register_buffer("candidate_x", candidate_embeddings.clone())
        nn.init.normal_(self.w_v.weight, mean=0.0, std=0.02)

    def forward(self, x, mask_index):
        v = self.w_v(x)
        h_mask = v.mean(dim=0)
        candidate_v = self.w_v(self.candidate_x)
        logits = F.cosine_similarity(
            h_mask.unsqueeze(0),
            candidate_v,
            dim=-1,
        )
        return {
            "logits": logits,
            "h_mask": h_mask,
            "candidate_v": candidate_v,
        }

set_seed(SEED)
uniform_model = UniformAttentionClassifier(
    candidate_embeddings=candidate_x,
    input_dim=embedding_dim,
    attention_dim=4,
).to(DEVICE)

uniform_history = fit_model(
    uniform_model,
    train_data,
    valid_data,
    steps=200,
    learning_rate=5e-3,
)

main_train = dataset_metrics(model, train_data)
main_valid = dataset_metrics(model, valid_data)
uniform_train = dataset_metrics(uniform_model, train_data)
uniform_valid = dataset_metrics(uniform_model, valid_data)

ablation_df = pd.DataFrame([
    {
        "模型": "学習するSelf-Attention",
        "学習可能な数": sum(p.numel() for p in model.parameters()),
        "学習正解率": main_train[1],
        "検証正解率": main_valid[1],
    },
    {
        "模型": "均等Attention対照",
        "学習可能な数": sum(p.numel() for p in uniform_model.parameters()),
        "学習正解率": uniform_train[1],
        "検証正解率": uniform_valid[1],
    },
])

display(
    ablation_df.style
    .format({
        "学習可能な数": "{:,}",
        "学習正解率": "{:.1%}",
        "検証正解率": "{:.1%}",
    })
)


step   1: 学習損失=1.0909, 学習正解率=33.3%, 検証損失=1.0954, 検証正解率=33.3%
step 100: 学習損失=0.3690, 学習正解率=100.0%, 検証損失=0.4036, 検証正解率=100.0%
step 200: 学習損失=0.3690, 学習正解率=100.0%, 検証損失=0.4036, 検証正解率=100.0%

検証損失が最小だったstep 35の状態を採用しました。


,模型,学習可能な数,学習正解率,検証正解率
0,学習するSelf-Attention,"9,216",100.0%,100.0%
1,均等Attention対照,"3,072",100.0%,100.0%


In [24]:
def uniform_prediction(current_model, text):
    item = encode_text(text)
    current_model.eval()
    with torch.no_grad():
        out = current_model(item["x"], item["mask_index"])
        ratios = torch.softmax(out["logits"], dim=0)

    rows = []
    for i, word in enumerate(CANDIDATES):
        rows.append({
            "候補": word,
            "コサイン得点": float(out["logits"][i]),
            "三候補内の比率": float(ratios[i]),
        })
    return pd.DataFrame(rows).sort_values("三候補内の比率", ascending=False)

print("均等Attention対照による指定文の候補順位")
display(
    uniform_prediction(uniform_model, DEMO_TEXT).style
    .format({
        "コサイン得点": "{:.5f}",
        "三候補内の比率": "{:.1%}",
    })
    .bar(subset=["三候補内の比率"], vmin=0, vmax=1, color="#c7a4ff")
)


均等Attention対照による指定文の候補順位


,候補,コサイン得点,三候補内の比率
0,好き,0.96776,69.2%
2,悩む,-0.37082,18.1%
1,寝る,-0.72831,12.7%


均等Attention対照の成績が高かった場合、結論は次のようになります。

> この小さな三分類は、事前学習済みEmbeddingと \(W_V\) だけでも解きやすい。したがって、Self-Attentionの計算経路は観察できたが、学習されたAttentionが正解に不可欠だったとは言えない。

これは教材の失敗ではありません。  
むしろ、「Attentionの棒グラフを、そのまま模型の理由と呼んではいけない」ことを自分の実験で確認できます。

成績が低かった場合も、一回の乱数結果だけで必要性を断定しません。  
乱数シードやデータを変えて繰り返す必要があります。


## 12. 本物のLLMとは何が違うか

> **本書との対応（紙面ページ）**  
> 186〜193ページ：Transformer全体の部品と、Embeddingから出力候補までの流れ。

この模型は、Transformer系言語モデルの計算の一部を切り出した教材模型です。  
実際のGPT系LLMへ近づくには、少なくとも次の部品と処理が必要です。

| このNotebook | 実際のGPT系LLMへ向けて必要なもの |
|---|---|
| <code>[MASK]</code>位置を一回分類 | <code>[MASK]</code>を使わず、直前までから次トークンを予測 |
| 三候補だけを比較 | 語彙全体へlogitを出す |
| 一層・一ヘッド | 多層・Multi-Head Attention |
| 位置情報なし | 位置情報を加える |
| FFNなし | 各位置を変換するFFN |
| 残差接続・LayerNormなし | 残差接続とLayer Normalization |
| 一回で終了 | 選んだトークンを入力へ戻して生成を繰り返す |
| 30件の教師あり三分類 | 大規模コーパスによる次トークン学習 |

実際のBERTの穴埋めでは、単語・位置・文種のEmbeddingを組み合わせ、Encoderを何層も通し、MLM用の出力層から語彙全体の得点を出します。  
このNotebookは、その経路を使わず、未文脈化の単語Embeddingを教材模型へ直接入れています。

また、BERTは左右の文脈を使うEncoder型で、GPT系LLMは未来側を見ない因果マスクを使うDecoder型です。  
このNotebookはBERTのTokenizerとEmbedding表を借りていますが、BERT本体でもGPT本体でもありません。

したがって、記事では次のように紹介できます。

> Transformer系LLMにも使われる、Embedding、Q–K内積、softmax、Valueの重み付き平均という計算の一部を、小さな模型で追体験する。



## 13. 自由研究の課題

### 必修

1. 対象語を10個考え、実行前に三候補の一位を予想する。
2. 各入力の一位、比率、最大Attentionのトークンを記録する。
3. 予想と違った例を一つ選び、Embedding、学習例、模型の省略部分から理由を考える。
4. 「この実験だけでは言えないこと」を一文で書く。

下のセルの文字列を書き換えて実験できます。


In [25]:
MY_TEXT = "私がチョコレートが[MASK]"

# [MASK]を一つだけ残して、自由に書き換えてください。
_ = show_result(model, MY_TEXT)


入力: 私がチョコレートが[MASK]
トークン: ['私', 'が', 'チョコレート', 'が', '[MASK]']
[MASK]位置のQuery: [-0.5747 -0.5547  0.5706 -0.5809]
[MASK]位置の文脈ベクトル: [-0.0368 -0.0918 -0.0535 -0.0878]

【Q・KからAttentionを作る】


,位置,トークン,内積 q・k,÷ √4 後,Attention
0,0,私,0.03357,0.01679,25.3%
1,1,が,-2.03174,-1.01587,9.0%
2,2,チョコレート,0.86342,0.43171,38.3%
3,3,が,-2.03174,-1.01587,9.0%
4,4,[MASK],-0.59079,-0.29540,18.5%


Attention合計: 1.000000

【文脈ベクトルと三候補を比べる】


,候補,候補のトークン分割,コサイン得点,三候補内の比率
0,好き,好き,0.97196,68.3%
1,寝る,寝 / ##る,-0.33104,18.6%
2,悩む,悩 / ##む,-0.67436,13.2%


一位: 好き


### 発展

- Attention次元を4から8、16、32へ変えると結果はどう変わるか。
- 学習データから一種類ずつ語を減らすと何が変わるか。
- 乱数シードを0以外へ変えても同じ傾向になるか。
- 位置Embeddingを加えると、二個の「が」を区別できるか。
- FFN、残差接続、Layer Normalizationを一つずつ加えると、何が変わるか。
- 三候補ではなく語彙全体へ得点を出すには、どんな行列が必要か。

結果が変わったときは、「どの部品を変えたか」と「何が変わったか」を分けて記録してください。


## 14. まとめ

このNotebookで追った計算は、次の一本道です。

$$
\text{トークン}
\rightarrow
\text{Embedding}
\rightarrow
Q,K,V
\rightarrow
\frac{QK^\top}{\sqrt{4}}
\rightarrow
\text{softmax}
\rightarrow
AV
\rightarrow
\text{三候補との比較}
$$

高校生向けに言い換えると、次のとおりです。

1. 言葉の断片を数の並びへ変える。
2. <code>[MASK]</code>が各トークンをどれくらい参照するか、QとKの掛け算で決める。
3. その割合でValueを混ぜ、文脈を表す4個の数を作る。
4. 「好き」「寝る」「悩む」も数の並びへ変え、文脈と向きが近い順に並べる。
5. 正解例とのずれが小さくなるよう、三個の変換行列を直す。

ここまででLLM全体を作ったわけではありません。  
しかし、Transformer系LLMの内部にもあるScaled Dot-Product Attentionの式を、ブラックボックスのAPIに任せず、自分で計算しました。

次の発展編では、今回わざと省いた位置情報、FFN、残差接続、Layer Normalization、因果マスク、語彙全体への出力を、必要な理由とともに一つずつ追加できます。


## 参考文献・出典

- Ashish Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), 2017. Scaled Dot-Product Attentionの出典。
- Jacob Devlin et al., [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805), 2018.
- 東北大学乾研究室, [tohoku-nlp/bert-base-japanese-v3 model card](https://huggingface.co/tohoku-nlp/bert-base-japanese-v3). Apache License 2.0、Tokenizer、学習データ、モデル構成の情報。
- Sarthak Jain and Byron C. Wallace, [Attention is not Explanation](https://arxiv.org/abs/1902.10186), 2019. Attention重みの解釈に関する問題提起。
- 中西崇文『ChatGPTはどのように動いているのか？』翔泳社、2026年。本企画の出発点。本文の転載はしていません。
